# LedaFlow Simulations with PID Control

In [1]:
using CSV
using DataFrames
lf_case_id = "c3af314f-320b-440e-b457-7e981721951b" # Ledaflow Case ID  
pid_ctrl_file = "ctrl_scheme_A.jl"

include(pid_ctrl_file)
include("lf_softshell.jl")
include("run_ledaflow_sim_script.jl")

# RUN LEDAFOW TO STEADY-STATE 
include("run_ledaflow_ss.jl")

Process(`'/mnt/c/Program Files/Kongsberg/LedaFlow Engineering v2.11.271.018/softsh.exe' '/home/archanak/Kumaraswamy_2024_2027/Ongoing Work/2026_NPC_Workshop/ledaflow_ss.js'`, ProcessExited(0))

In [2]:
##############################
# INITIALIZATION 
##############################
# Initialization for outputs 
outputs = [1.0, 1.0, 1.0, 1.0, 3458.0]

# Initialization for measurements
#### List of measurements from LedaFlow (order MUST match run_ledaflow_sim)
#  1. Mainline Flow Rate (19500 m)     8. Well 2 BHP
#  2. Mainline Pressure (500 m)        9. Well 3 Flow Rate
#  3. Well 1 Flow Rate                10. Well 3 Pressure
#  4. Well 1 Pressure                 11. Well 3 BHP
#  5. Well 1 BHP                      12. Well 4 Flow Rate
#  6. Well 2 Flow Rate                13. Well 4 Pressure
#  7. Well 2 Pressure                 14. Well 4 BHP

ss_output_csv_path = joinpath(@__DIR__, "ss_lf_output.csv")

df = CSV.read(ss_output_csv_path, DataFrame;
        header = 10,                 # column names on row 10
        skipto = 12,                 # skip the units row (11)
        delim = ',',                 # comma-delimited export
        decimal = '.',               # period decimals (e.g. 6.0000000)
        missingstring = ["--", ""],  # LedaFlow missing marker + trailing empty col
        normalizenames = false,      # keep names like "Pressure@Line 1 P 500m"
        types = Float64,
        )

mainline_pressure = df[end, "Pressure@Mainline P 500m"]
mainline_flow     = df[end, "MFR - total@Mainline P 19500m"]*(-1)   # match sim (19500 m)

well1_flow = df[end, "MFR - total liquid@Wellbore1 P MFR"]*(-1)
well2_flow = df[end, "MFR - total liquid@Wellbore2 P MFR"]*(-1)
well3_flow = df[end, "MFR - total liquid@Wellbore3 P MFR"]*(-1)
well4_flow = df[end, "MFR - total liquid@Wellbore4 P MFR"]*(-1)

well1_press = df[end, "Pressure@Wellbore1 P MFR"]
well2_press = df[end, "Pressure@Wellbore2 P MFR"]
well3_press = df[end, "Pressure@Wellbore3 P MFR"]
well4_press = df[end, "Pressure@Wellbore4 P MFR"]

well1_bhp = df[end, "BHP - Zone 1@Well 1"]
well2_bhp = df[end, "BHP - Zone 1@Well 2"]
well3_bhp = df[end, "BHP - Zone 1@Well 3"]
well4_bhp = df[end, "BHP - Zone 1@Well 4"]

old_measurements = [mainline_flow, mainline_pressure,
    well1_flow, well1_press, well1_bhp,
    well2_flow, well2_press, well2_bhp,
    well3_flow, well3_press, well3_bhp,
    well4_flow, well4_press, well4_bhp]
old_outputs = outputs
old_clamped_outputs = outputs
clamped_outputs = outputs
println("Initial Measurements: $old_measurements")
println("Initial Outputs: $old_outputs")


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


Initial Measurements: [500.24592, 197.81419, 125.04508, 150.79354, 245.04503, 125.04508, 150.79354, 245.04503, 125.04508, 150.79354, 245.04503, 125.04508, 150.79354, 245.04503]
Initial Outputs: [1.0, 1.0, 1.0, 1.0, 3458.0]


In [3]:
##############################
# RUN PID CONTROLLER EVER TIME STEP 
##############################
# Time step in seconds 
dt = 30
nt = 7200

for i = 0:dt:nt
    new_measurements = run_ledaflow_sim(clamped_outputs, old_clamped_outputs, dt, i)    

    # Update old outputs and old clamped outputs for the next PID run 
    old_outputs = outputs
    old_clamped_outputs = clamped_outputs

    outputs, clamped_outputs = run_pid_controller(new_measurements, old_measurements, old_outputs, old_clamped_outputs, dt, i)

    # Update old measurements and outputs for next time step
    old_measurements = new_measurements
    println("Time step: $i")
end 


choke_vlv_op_1: 1.0, choke_vlv_op_2: 1.0, choke_vlv_op_3: 1.0, choke_vlv_op_4: 1.0, pump_speed: 3458.0
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 1.0, choke_vlv_op_3_prev: 1.0, choke_vlv_op_4_prev: 1.0
FC2_error: -0.04520999999999731, FC3_error: -0.04520999999999731, FC4_error: -0.04520999999999731, PC_pump_error: -0.01553999999998723


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


FC2_OP: 0.9994207472002731, FC3_OP: 0.9994207472002731, FC4_OP: 0.9994207472002731, PC_pump_OP: 3457.9468617054763
outputs: [1.0, 0.9994207472002731, 0.9994207472002731, 0.9994207472002731, 3457.9468617054763], clamped_outputs: [1.0, 0.9994207472002731, 0.9994207472002731, 0.9994207472002731, 3457.9468617054763]
Time step: 0
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.9994207472002731, choke_vlv_op_3: 0.9994207472002731, choke_vlv_op_4: 0.9994207472002731, pump_speed: 3457.9468617054763
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 1.0, choke_vlv_op_3_prev: 1.0, choke_vlv_op_4_prev: 1.0
FC2_error: -0.035520000000005325, FC3_error: -0.03660000000000707, FC4_error: -0.03660000000000707, PC_pump_error: -0.016239999999982047
FC2_OP: 0.998966065657424, FC3_OP: 0.998952182173292, FC4_OP: 0.998952182173292, PC_pump_OP: 3457.912328993542
outputs: [1.0, 0.998966065657424, 0.998952182173292, 0.998952182173292, 3457.912328993542], clamped_outputs: [1.0, 0.998966065657424, 0.998952182173292, 0.9989521

┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


FC2_error: -0.028319999999993684, FC3_error: -0.03006000000000597, FC4_error: -0.03006000000000597, PC_pump_error: -0.01682999999999879
FC2_OP: 0.998603526835904, FC3_OP: 0.9985673216407579, FC4_OP: 0.9985673216407579, PC_pump_OP: 3457.8805426913605
outputs: [1.0, 0.998603526835904, 0.9985673216407579, 0.9985673216407579, 3457.8805426913605], clamped_outputs: [1.0, 0.998603526835904, 0.9985673216407579, 0.9985673216407579, 3457.8805426913605]
Time step: 60
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.998603526835904, choke_vlv_op_3: 0.9985673216407579, choke_vlv_op_4: 0.9985673216407579, pump_speed: 3457.8805426913605
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 0.998966065657424, choke_vlv_op_3_prev: 0.998952182173292, choke_vlv_op_4_prev: 0.998952182173292


...

FC2_OP: 0.8233069756184499, FC3_OP: 0.8069973445198806, FC4_OP: 0.8069973445198806, PC_pump_OP: 3526.5746746518603
outputs: [1.0, 0.8233069756184499, 0.8069973445198806, 0.8069973445198806, 3526.5746746518603], clamped_outputs: [1.0, 0.8233069756184499, 0.8069973445198806, 0.8069973445198806, 3526.5746746518603]
Time step: 7140
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.8233069756184499, choke_vlv_op_3: 0.8069973445198806, choke_vlv_op_4: 0.8069973445198806, pump_speed: 3526.5746746518603
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 0.823288780771813, choke_vlv_op_3_prev: 0.8069781246836437, choke_vlv_op_4_prev: 0.8069781246836437


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


FC2_error: 0.0014400000000023283, FC3_error: 0.0015300000000024738, FC4_error: 0.0015300000000024738, PC_pump_error: -0.05875000000000341
FC2_OP: 0.823325426285408, FC3_OP: 0.8070169487272176, FC4_OP: 0.8070169487272176, PC_pump_OP: 3526.5603068258674
outputs: [1.0, 0.823325426285408, 0.8070169487272176, 0.8070169487272176, 3526.5603068258674], clamped_outputs: [1.0, 0.823325426285408, 0.8070169487272176, 0.8070169487272176, 3526.5603068258674]
Time step: 7170
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.823325426285408, choke_vlv_op_3: 0.8070169487272176, choke_vlv_op_4: 0.8070169487272176, pump_speed: 3526.5603068258674
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 0.8233069756184499, choke_vlv_op_3_prev: 0.8069973445198806, choke_vlv_op_4_prev: 0.8069973445198806


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


FC2_error: 0.0014699999999976399, FC3_error: 0.0015499999999946112, FC4_error: 0.0015499999999946112, PC_pump_error: -0.05753999999998882
FC2_OP: 0.8233442617505449, FC3_OP: 0.8070368087548756, FC4_OP: 0.8070368087548756, PC_pump_OP: 3526.546971142686
outputs: [1.0, 0.8233442617505449, 0.8070368087548756, 0.8070368087548756, 3526.546971142686], clamped_outputs: [1.0, 0.8233442617505449, 0.8070368087548756, 0.8070368087548756, 3526.546971142686]
Time step: 7200


In [4]:
# EXTRACT ALL RESULTS FROM LEDAFLOW TO CSV FILE
include("extract_full_output.jl")

extract_full_output(lf_case_id, "caseA_trends.csv")

Process(`'/mnt/c/Program Files/Kongsberg/LedaFlow Engineering v2.11.271.018/softsh.exe' '/home/archanak/Kumaraswamy_2024_2027/Ongoing Work/2026_NPC_Workshop/ledaflow_extract.js'`, ProcessExited(0))